## 2. Load and Inspect the Survival Dataset

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the original project folder.
PROJECT = Path.cwd().resolve()

if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

if PROJECT.name == "bridge_time_primary_notebooks":
    PROJECT = PROJECT.parent

# Define the existing dataset location.
FINAL = PROJECT / "data" / "final"
SURVIVAL_PATH = FINAL / "bridge_survival_dataset.parquet"

print("Project directory:", PROJECT)
print("Dataset exists:", SURVIVAL_PATH.exists())
print("Dataset path:", SURVIVAL_PATH)

Project directory: /Users/annie/Downloads/bridge-deterioration
Dataset exists: True
Dataset path: /Users/annie/Downloads/bridge-deterioration/data/final/bridge_survival_dataset.parquet


In [10]:
# Load the prepared survival dataset.
survival_data = pd.read_parquet(SURVIVAL_PATH)

# Verify that the data loaded successfully.
print("Total records:", f"{len(survival_data):,}")
print("Columns:", survival_data.columns.tolist())

# Preview the dataset.
display(survival_data.head())

Total records: 521,563
Columns: ['bridge_id', 'start_year', 'bridge_age', 'YEAR_BUILT_027', 'YEAR_RECONSTRUCTED_106', 'ADT_029', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B', 'STRUCTURE_LEN_MT_049', 'end_year', 'event', 'stop_reason', 'last_year', 'duration']


,bridge_id,start_year,bridge_age,YEAR_BUILT_027,YEAR_RECONSTRUCTED_106,ADT_029,STRUCTURE_KIND_043A,STRUCTURE_TYPE_043B,STRUCTURE_LEN_MT_049,end_year,event,stop_reason,last_year,duration
0,01_00000000000S700,1996,7.0,1989.0,0.0,80.0,5,01,11.0,1999.0,1,Deterioration,2005,3.0
1,01_00000000000S702,2003,4.0,1999.0,0.0,50.0,5,05,14.7,2022.0,1,Deterioration,2025,19.0
2,01_00000000000S703,2006,4.0,2002.0,0.0,150.0,5,01,13.7,2022.0,1,Deterioration,2025,16.0
3,01_0000000000M0022,1998,56.0,1942.0,0.0,800.0,1,02,43.3,2006.0,1,Deterioration,2025,8.0
4,01_00000BLN0BR0010,2012,38.0,1974.0,0.0,250.0,1,19,6.4,2015.0,1,Deterioration,2025,3.0


## 3. Temporal Train/Test Split Design

In [11]:


# Define the chronological training and testing periods.
TRAIN_START = 1992
TRAIN_END = 2020

TEST_START = 2021
TEST_END = 2025

# Display the selected periods.
print(f"Training: {TRAIN_START}–{TRAIN_END}")
print(f"Training years: {TRAIN_END - TRAIN_START + 1}")

print(f"\nTesting: {TEST_START}–{TEST_END}")
print(f"Testing years: {TEST_END - TEST_START + 1}")

print("\nAvailable baseline years:")
print("Earliest:", survival_data["start_year"].min())
print("Latest:", survival_data["start_year"].max())


Training: 1992–2020
Training years: 29

Testing: 2021–2025
Testing years: 5

Available baseline years:
Earliest: 1992
Latest: 2024


In [12]:

# Count survival records by their first observed Good year.
baseline_counts = (
    survival_data.groupby("start_year")
    .size()
    .rename("bridge_count")
    .reset_index()
)

# Summarise baseline availability across the two periods.
training_baselines = survival_data[
    survival_data["start_year"].between(TRAIN_START, TRAIN_END)
]

testing_baselines = survival_data[
    survival_data["start_year"].between(TEST_START, TEST_END)
]

print("Training-period baselines:",
      f"{len(training_baselines):,}")

print("Testing-period baselines:",
      f"{len(testing_baselines):,}")

print("\nBaseline counts by year:")
display(baseline_counts)


Training-period baselines: 503,406
Testing-period baselines: 18,157

Baseline counts by year:


,start_year,bridge_count
0,1992,253945
1,1993,27187
2,1994,6186
3,1995,9630
4,1996,5603
5,1997,14631
6,1998,5096
7,1999,5694
8,2000,12426
9,2001,7799


## 4. Construct Training and Testing Datasets

In [13]:

# Select bridges with baselines in the training period.
train_data = survival_data.loc[
    survival_data["start_year"].between(TRAIN_START, TRAIN_END)
].copy()

# Censor outcomes that occur after the training cutoff.
after_cutoff = train_data["end_year"] > TRAIN_END

train_data.loc[after_cutoff, "end_year"] = TRAIN_END
train_data.loc[after_cutoff, "event"] = 0
train_data.loc[after_cutoff, "stop_reason"] = "Training cutoff"

# Recalculate observed survival durations.
train_data["duration"] = (
    train_data["end_year"] - train_data["start_year"]
)

# Remove records without positive follow-up.
train_data = train_data.loc[
    train_data["duration"] > 0
].copy()

print("Training records:", f"{len(train_data):,}")
print("Observed deterioration events:",
      f"{train_data['event'].sum():,}")
print("Censored records:",
      f"{(train_data['event'] == 0).sum():,}")

print("\nLatest training endpoint:", train_data["end_year"].max())
print("Minimum training duration:", train_data["duration"].min())


Training records: 498,875
Observed deterioration events: 237,197
Censored records: 261,678

Latest training endpoint: 2020.0
Minimum training duration: 1.0


In [14]:

# Identify bridges whose original survival interval reaches 2021.
test_candidates = survival_data.loc[
    (survival_data["start_year"] <= TEST_START) &
    (survival_data["end_year"] >= TEST_START)
].copy()

# Load bridge identifiers observed in the first testing year.
observed_2021 = pd.read_parquet(
    PROJECT / "data" / "processed" / "nbi_2021_clean.parquet",
    columns=["bridge_id", "condition"]
)

# Retain bridges observed in Good condition in 2021.
good_2021_ids = set(
    observed_2021.loc[
        observed_2021["condition"] == "Good",
        "bridge_id"
    ]
)

test_candidates = test_candidates.loc[
    test_candidates["bridge_id"].isin(good_2021_ids)
].copy()

print("Eligible 2021 test candidates:", f"{len(test_candidates):,}")
print("Duplicate bridge IDs:", test_candidates["bridge_id"].duplicated().sum())

print("\nOriginal baseline years:")
display(test_candidates["start_year"].describe())


Eligible 2021 test candidates: 172,149
Duplicate bridge IDs: 0

Original baseline years:


count    172149.000000
mean       2005.032786
std          10.021438
min        1992.000000
25%        1993.000000
50%        2006.000000
75%        2014.000000
max        2021.000000
Name: start_year, dtype: float64

In [15]:

# Inspect the survival endpoints of the 2021 test candidates.
print("Candidate records:", f"{len(test_candidates):,}")

print("\nOriginal endpoint years:")
display(
    test_candidates["end_year"]
    .value_counts()
    .sort_index()
)

print("\nCandidates ending in 2021:")
display(
    test_candidates.loc[
        test_candidates["end_year"] == TEST_START,
        "stop_reason"
    ].value_counts()
)

print("\nCandidates with follow-up beyond 2021:")
print(
    f"{(test_candidates['end_year'] > TEST_START).sum():,}"
)

print("\nCandidates with original baselines in 2021:")
print(
    f"{(test_candidates['start_year'] == TEST_START).sum():,}"
)


Candidate records: 172,149

Original endpoint years:


end_year
2021.0       383
2022.0      6092
2023.0      5661
2024.0      5606
2025.0    154407
Name: count, dtype: int64


Candidates ending in 2021:


stop_reason
End of observation    183
Structure change      170
Observation gap        30
Name: count, dtype: int64


Candidates with follow-up beyond 2021:
171,766

Candidates with original baselines in 2021:
4,611


In [16]:

# Select bridges with follow-up beyond the 2021 test baseline.
test_data = test_candidates.loc[
    test_candidates["end_year"] > TEST_START
].copy()

# Record the testing baseline and prior survival time.
test_data["test_start_year"] = TEST_START

test_data["entry_duration"] = (
    test_data["test_start_year"] - test_data["start_year"]
)

# Calculate follow-up time during the testing period.
test_data["test_duration"] = (
    test_data["end_year"] - test_data["test_start_year"]
)

# Preserve the original event and censoring indicators.
test_data["test_event"] = test_data["event"]

print("Provisional test records:", f"{len(test_data):,}")
print("Test deterioration events:", f"{test_data['test_event'].sum():,}")
print("Test censored records:",
      f"{(test_data['test_event'] == 0).sum():,}")

print("\nPrior survival time:")
display(test_data["entry_duration"].describe())

print("\nTesting follow-up duration:")
display(test_data["test_duration"].describe())


Provisional test records: 171,766
Test deterioration events: 22,396
Test censored records: 149,370

Prior survival time:


count    171766.000000
mean         15.972137
std          10.019869
min           0.000000
25%           7.000000
50%          15.000000
75%          28.000000
max          29.000000
Name: entry_duration, dtype: float64


Testing follow-up duration:


count    171766.000000
mean          3.795047
std           0.664580
min           1.000000
25%           4.000000
50%           4.000000
75%           4.000000
max           4.000000
Name: test_duration, dtype: float64

In [17]:

# Select predictor values recorded at the 2021 testing baseline.
test_predictor_columns = [
    "bridge_id",
    "bridge_age",
    "YEAR_BUILT_027",
    "YEAR_RECONSTRUCTED_106",
    "ADT_029",
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B",
    "STRUCTURE_LEN_MT_049"
]

predictors_2021 = pd.read_parquet(
    PROJECT / "data" / "processed" / "nbi_2021_clean.parquet",
    columns=test_predictor_columns
)

# Match the 2021 predictors to the provisional testing cohort.
test_2021 = test_data[
    ["bridge_id", "start_year", "end_year",
     "event", "entry_duration", "test_duration"]
].merge(
    predictors_2021,
    on="bridge_id",
    how="inner",
    validate="one_to_one"
)

print("Testing records with 2021 predictors:", f"{len(test_2021):,}")
print("Duplicate bridge identifiers:",
      test_2021["bridge_id"].duplicated().sum())

print("\nMissing predictor values:")
display(
    test_2021[
        [
            "bridge_age",
            "ADT_029",
            "STRUCTURE_LEN_MT_049",
            "STRUCTURE_KIND_043A",
            "STRUCTURE_TYPE_043B"
        ]
    ].isna().sum()
)



Testing records with 2021 predictors: 171,766
Duplicate bridge identifiers: 0

Missing predictor values:


bridge_age              2
ADT_029                 0
STRUCTURE_LEN_MT_049    0
STRUCTURE_KIND_043A     0
STRUCTURE_TYPE_043B     0
dtype: int64

In [18]:

# Preserve the original survival time scale for test evaluation.
test_2021["duration"] = (
    test_2021["end_year"] - test_2021["start_year"]
)

# Record the prediction year and time already survived.
test_2021["prediction_year"] = TEST_START

# Keep the event indicator consistent with the training dataset.
test_2021["event"] = test_2021["event"].astype(int)

# Verify the test survival intervals.
print("Testing records:", f"{len(test_2021):,}")
print("Invalid total durations:",
      (test_2021["duration"] <= 0).sum())
print("Invalid entry durations:",
      (test_2021["entry_duration"] < 0).sum())
print("Events during testing:",
      f"{test_2021['event'].sum():,}")

print("\nTest time variables:")
display(
    test_2021[
        [
            "start_year",
            "prediction_year",
            "entry_duration",
            "test_duration",
            "duration",
            "event"
        ]
    ].head(10)
)


Testing records: 171,766
Invalid total durations: 0
Invalid entry durations: 0
Events during testing: 22,396

Test time variables:


,start_year,prediction_year,entry_duration,test_duration,duration,event
0,2003,2021,18,1.0,19.0,1
1,2006,2021,15,1.0,16.0,1
2,1993,2021,28,4.0,32.0,0
3,1993,2021,28,4.0,32.0,0
4,1993,2021,28,2.0,30.0,1
5,1993,2021,28,4.0,32.0,0
6,1993,2021,28,4.0,32.0,0
7,1993,2021,28,4.0,32.0,0
8,1993,2021,28,4.0,32.0,0
9,1993,2021,28,4.0,32.0,0


## 5. Validate the Temporal Split

In [19]:

# Check the temporal boundaries and shared bridge identifiers.
shared_ids = set(train_data["bridge_id"]) & set(test_2021["bridge_id"])

print("Training records:", f"{len(train_data):,}")
print("Testing records:", f"{len(test_2021):,}")
print("Bridge IDs present in both periods:", f"{len(shared_ids):,}")

print("\nTraining endpoint checks:")
print("Endpoints after 2020:", (train_data["end_year"] > TRAIN_END).sum())
print("Non-positive durations:", (train_data["duration"] <= 0).sum())

print("\nTesting interval checks:")
print("Endpoints after 2025:", (test_2021["end_year"] > TEST_END).sum())
print("Non-positive test durations:", (test_2021["test_duration"] <= 0).sum())
print(
    "Inconsistent total durations:",
    (
        test_2021["duration"] !=
        test_2021["entry_duration"] + test_2021["test_duration"]
    ).sum()
)

print("\nTraining and testing event counts:")
print("Training:", f"{train_data['event'].sum():,}")
print("Testing:", f"{test_2021['event'].sum():,}")


Training records: 498,875
Testing records: 171,766
Bridge IDs present in both periods: 162,714

Training endpoint checks:
Endpoints after 2020: 0
Non-positive durations: 0

Testing interval checks:
Endpoints after 2025: 0
Non-positive test durations: 0
Inconsistent total durations: 0

Training and testing event counts:
Training: 237,197
Testing: 22,396


In [20]:

# Compare training endpoints with their original survival endpoints.
training_check = train_data[
    ["bridge_id", "end_year", "event"]
].merge(
    survival_data[
        ["bridge_id", "end_year", "event"]
    ],
    on="bridge_id",
    suffixes=("_train", "_original"),
    validate="one_to_one"
)

# Verify that training endpoints were only shortened when necessary.
incorrect_endpoints = (
    training_check["end_year_train"] !=
    training_check["end_year_original"].clip(upper=TRAIN_END)
)

# Verify that events after 2020 were converted to censoring.
expected_events = (
    (training_check["event_original"] == 1) &
    (training_check["end_year_original"] <= TRAIN_END)
).astype(int)

print("Incorrect training endpoints:", incorrect_endpoints.sum())
print(
    "Incorrect training event indicators:",
    (training_check["event_train"] != expected_events).sum()
)

print("\nTraining endpoints:")
display(train_data["end_year"].describe())


Incorrect training endpoints: 0
Incorrect training event indicators: 0

Training endpoints:


count    498875.000000
mean       2009.841181
std           9.824349
min        1993.000000
25%        2000.000000
50%        2012.000000
75%        2020.000000
max        2020.000000
Name: end_year, dtype: float64

In [21]:

# Define the predictors shared by all four models.
predictor_columns = [
    "bridge_age",
    "ADT_029",
    "STRUCTURE_LEN_MT_049",
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B"
]

# Compare predictor availability in training and testing.
predictor_check = pd.DataFrame({
    "training_missing": train_data[predictor_columns].isna().sum(),
    "testing_missing": test_2021[predictor_columns].isna().sum(),
    "training_dtype": train_data[predictor_columns].dtypes.astype(str),
    "testing_dtype": test_2021[predictor_columns].dtypes.astype(str)
})

display(predictor_check)


,training_missing,testing_missing,training_dtype,testing_dtype
bridge_age,1449,2,float64,float64
ADT_029,2003,0,float64,int64
STRUCTURE_LEN_MT_049,230,0,float64,float64
STRUCTURE_KIND_043A,348,0,object,object
STRUCTURE_TYPE_043B,389,0,object,object


In [22]:

# Standardise numerical predictors across both datasets.
numeric_predictors = [
    "bridge_age",
    "ADT_029",
    "STRUCTURE_LEN_MT_049"
]

categorical_predictors = [
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B"
]

for column in numeric_predictors:
    train_data[column] = pd.to_numeric(
        train_data[column], errors="coerce"
    ).astype("float64")

    test_2021[column] = pd.to_numeric(
        test_2021[column], errors="coerce"
    ).astype("float64")

# Preserve structure classifications as categorical strings.
for column in categorical_predictors:
    train_data[column] = train_data[column].astype("string")
    test_2021[column] = test_2021[column].astype("string")

print("Training predictor types:")
display(train_data[predictor_columns].dtypes)

print("\nTesting predictor types:")
display(test_2021[predictor_columns].dtypes)


Training predictor types:


bridge_age                     float64
ADT_029                        float64
STRUCTURE_LEN_MT_049           float64
STRUCTURE_KIND_043A     string[python]
STRUCTURE_TYPE_043B     string[python]
dtype: object


Testing predictor types:


bridge_age                     float64
ADT_029                        float64
STRUCTURE_LEN_MT_049           float64
STRUCTURE_KIND_043A     string[python]
STRUCTURE_TYPE_043B     string[python]
dtype: object

## 6. Export Shared Datasets

In [23]:

# Export the shared training and testing datasets.
FINAL.mkdir(parents=True, exist_ok=True)

train_path = FINAL / "bridge_survival_train.parquet"
test_path = FINAL / "bridge_survival_test.parquet"

train_data.to_parquet(train_path, index=False)
test_2021.to_parquet(test_path, index=False)

# Confirm that both files were saved.
print("Training file:", train_path)
print("Training records:", f"{len(train_data):,}")

print("\nTesting file:", test_path)
print("Testing records:", f"{len(test_2021):,}")

print("\nBoth files exist:", train_path.exists() and test_path.exists())


Training file: /Users/annie/Downloads/bridge-deterioration/data/final/bridge_survival_train.parquet
Training records: 498,875

Testing file: /Users/annie/Downloads/bridge-deterioration/data/final/bridge_survival_test.parquet
Testing records: 171,766

Both files exist: True
